In [ ]:
cfg.actor.model.

namespace(model_path='/home/ubuntu/cache/models/RLinf-Pi0-LIBERO-Spatial-Object-Goal-SFT',
          num_action_chunks=5,
          add_value_head=True,
          openpi=namespace(detach_critic_input=True,
                           config_name='pi0_libero',
                           num_images_in_input=2,
                           noise_level=0.5,
                           action_chunk=5),
          model_type=<SupportedModel.OPENPI: 'openpi'>,
          precision=None)

In [35]:
import yaml
from types import SimpleNamespace
from omegaconf.omegaconf import OmegaConf
import sys
sys.path.append('/home/ubuntu/haitong-south-2/RLinf')
from rlinf.models import get_model
from rlinf.config import SupportedModel

def dict_to_namespace(d):
    """Recursively converts a dictionary and its nested dictionaries to SimpleNamespace."""
    if not isinstance(d, dict):
        return d
    # Convert the current dictionary to a SimpleNamespace and recurse on its values
    return SimpleNamespace(**{k: dict_to_namespace(v) for k, v in d.items()})
config_path = '/home/ubuntu/haitong-south-2/RLinf/examples/embodiment/config/libero_spatial_ppo_openpi_quickstart.yaml'
with open(config_path, 'r') as f:
    cfg = yaml.safe_load(f)

cfg = OmegaConf.create(cfg)
cfg.actor.model.model_type = SupportedModel.OPENPI
cfg.actor.model.precision = None
cfg.actor.model.is_lora = False
cfg.actor.model.num_action_chunks = 5
cfg.actor.model.action_dim = 7
cfg.actor.model.openpi.config_name = "pi0_libero"
cfg.actor.model.openpi.num_images_in_input = 2
cfg.actor.model.openpi.noise_level = 0.5
cfg.actor.model.openpi.action_chunk = cfg.actor.model.num_action_chunks
cfg.actor.model.openpi.num_steps = 4
cfg.actor.model.openpi.train_expert_only = True
cfg.actor.model.openpi.action_env_dim = 7
cfg.actor.model.openpi.noise_method = "flow_sde"
cfg.actor.model.openpi.add_value_head = cfg.actor.model.add_value_head
cfg.actor.model.openpi.detach_critic_input = True


In [36]:
model = get_model(cfg.actor.model)

INFO:root:Loaded norm stats from /home/ubuntu/cache/models/RLinf-Pi0-LIBERO-Spatial-Object-Goal-SFT/physical-intelligence/libero
INFO:root:Loaded norm stats from /home/ubuntu/cache/models/RLinf-Pi0-LIBERO-Spatial-Object-Goal-SFT/physical-intelligence/libero


In [39]:
model.device

AttributeError: 'OpenPi0ForRLActionPrediction' object has no attribute 'device'

In [ ]:
# create dummy input
import torch
bs = 8
channels = 3
size = 224
token_max_length = 4
state_dim = 32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

images = [torch.randn(bs, channels, size, size, device=device)] * 3
img_masks = [torch.ones(bs, device=device)] * 3
lang_tokens = torch.randint(0, 10000, (bs, token_max_length), device=device)
lang_masks = torch.ones(bs, token_max_length, device=device)
state = torch.randn(bs, state_dim)


prefix_embs, prefix_pad_masks, prefix_att_masks = model.embed_prefix(
            images, img_masks, lang_tokens, lang_masks
        )

In [44]:
print(prefix_embs.shape)
print(prefix_pad_masks.shape)
print(prefix_att_masks.shape)

torch.Size([8, 772, 2048])
torch.Size([8, 772])
torch.Size([8, 772])


In [ ]:
from openpi.models_pytorch.pi0_pytorch import PI0Pytorch, make_att_2d_masks
